In [11]:
import run as run
import chephy_model as cpm
import GNN_model
import gnn_utils
import numpy as np
import torch
import pandas as pd
from torch_geometric.transforms import ToUndirected


In [12]:
# import matplotlib.pyplot as plt

# # Flatten the matrix and plot histogram
# distances = chephy_matrix.flatten()
# plt.figure(figsize=(10, 6))
# plt.hist(distances, bins=50, edgecolor='black')
# plt.xlabel('Distance')
# plt.xlabel('Distance')
# plt.ylabel('Frequency')
# plt.title('Distribution of Distances in Chephy Matrix')
# plt.grid(axis='y', alpha=0.3)
# plt.show()

In [13]:
def create_hetero_data(data, label_header, cdr3_header, atchley_dict, n=1000, distance_threshold = 0.2, use_mst=True, enrich=False):

    data = data.sample(n=n, random_state=42)

    data = run.prepare_data(data, cdr3_header, label_header)    
    data = cpm.truncate_sequences(data, cdr3_header, 4, 4)

    sequences = set(data["cdr3_truncated"])
    epitopes = list(set(data[label_header]))

    chephy_matrix, sequences_list = cpm.find_che_phy_dist(sequences)
    sequence_indices = np.arange(len(sequences_list))

    data['tcr_index'] = data['cdr3_truncated'].apply(lambda x: sequences_list.index(x) if x in sequences_list else -1)
    data['epitope_index'] = data[label_header].apply(lambda x: epitopes.index(x) if x in epitopes else -1)
    print("Done indexing")

    data_hetero, labels = gnn_utils.build_hetero_graph(
                            data,
                            chephy_matrix,
                            sequences_list,
                            epitopes,
                            atchley_dict,
                            distance_threshold=distance_threshold,
                            use_mst=use_mst,
                            enrich=enrich  
                        )
    data_hetero = ToUndirected()(data_hetero)
    return data_hetero, data


In [14]:
atchley_path = "/home/dsi/chenbis/repos/sol_lab/files/atchley.csv"
atchley_df = pd.read_csv(atchley_path)
atchley_dict = atchley_df.set_index("amino.acid").to_dict(orient="index")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [15]:
import os

distance_threshold=0.08
use_mst=True
enrich=True

save_model_path = f"/home/dsi/chenbis/repos/sol_lab/tcr_epitope_prediction_gnn_mst/output/models/{distance_threshold if enrich else ''}_{'mst' if use_mst else ''}_{'enrich' if enrich else ''}"
model_type = "min"
os.makedirs(f"{save_model_path}/results/min", exist_ok=True)
os.makedirs(f"{save_model_path}/results/max", exist_ok=True)

In [16]:
input_file = "/home/dsi/chenbis/repos/sol_lab/data/train.tsv"
data = run.load_dataframe(input_file)
label_header = "peptide"
cdr3_header = "cdr3"

n = 50000

data_hetero_train, data_train = create_hetero_data(data, label_header, cdr3_header, atchley_dict, n=n, distance_threshold = distance_threshold, use_mst=use_mst, enrich=enrich)

KeyboardInterrupt: 

In [ ]:
input_file = "/home/dsi/chenbis/repos/sol_lab/data/test.tsv"
data = run.load_dataframe(input_file)
label_header = "peptide"
cdr3_header = "cdr3"
n = 15000

data_hetero_test, data_test = create_hetero_data(data, label_header, cdr3_header, atchley_dict, n=n, distance_threshold = distance_threshold, use_mst=use_mst, enrich=enrich)

Done indexing
computing MST


Using cache found in /home/dsi/chenbis/.cache/torch/hub/facebookresearch_esm_main


enrichment


In [ ]:
model = GNN_model.HeteroTCR(data_hetero_train.metadata(), 256, 3, "SAGE")
model = model.to(device)
data_hetero_train = data_hetero_train.to(device)
data_hetero_test = data_hetero_test.to(device)

In [ ]:
# Create edge label indices for training data
train_edge_pairs = data_train[['tcr_index', 'epitope_index']].values
train_edge_label_index = torch.tensor(train_edge_pairs.T)

# Create edge label indices for test data
test_edge_pairs = data_test[['tcr_index', 'epitope_index']].values
test_edge_label_index = torch.tensor(test_edge_pairs.T)

# Create labels
y_train = data_train['Binding'].values
y_test = data_test['Binding'].values

In [ ]:
import os
import torch
import pandas as pd


# save_model_path = f'/home/dsi/chenbis/repos/sol_lab/tcr_epitope_prediction_gnn_mst/output/models/{run.get_time()}'
os.makedirs(save_model_path, exist_ok=True)
NUM_EPOCHS = 1000

# Track performance
epoch = []
loss = []
acc = []
auc_roc = []
val_loss = []
val_acc = []
val_auc_roc = []

# Best model tracking
best_val_auc = -float('inf')
best_val_loss = float('inf')
best_auc_model_state = None
best_loss_model_state = None
best_auc_epoch = 0
best_loss_epoch = 0

optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=0.0)

for ep in range(1, NUM_EPOCHS + 1):
    train_loss, train_binary_accuracy, train_ROCAUC, valid_loss, val_binary_accuracy, val_ROCAUC = GNN_model.train(
        model, optimizer,
        data_hetero_train, data_hetero_test,
        train_edge_label_index, test_edge_label_index,
        y_train, y_test,
        device
    )

    print(
        'Train epoch: {} - loss: {:.4f} - binary_accuracy: {:.4f} - ROCAUC: {:.4f} - val_loss: {:.4f} - val_binary_accuracy: {:.4f} - val_ROCAUC: {:.4f}'.format(
            ep, train_loss.item(), train_binary_accuracy, train_ROCAUC, valid_loss, val_binary_accuracy, val_ROCAUC))

    # Save best AUC model
    if val_ROCAUC > best_val_auc:
        best_val_auc = val_ROCAUC
        best_auc_model_state = model.state_dict()
        best_auc_epoch = ep

    # Save best loss model
    if valid_loss < best_val_loss:
        best_val_loss = valid_loss
        best_loss_model_state = model.state_dict()
        best_loss_epoch = ep

    # Store metrics
    epoch.append(ep)
    loss.append(train_loss.item())
    acc.append(train_binary_accuracy.item())
    auc_roc.append(train_ROCAUC.item())
    val_loss.append(valid_loss.item())
    val_acc.append(val_binary_accuracy.item())
    val_auc_roc.append(val_ROCAUC.item())

# Save training history
root_history = '/home/dsi/chenbis/repos/sol_lab/output'
dfhistory = {
    'epoch': epoch,
    'loss': loss, 'acc': acc, 'auc_roc': auc_roc,
    'val_loss': val_loss, 'val_acc': val_acc, 'val_auc_roc': val_auc_roc
}
df = pd.DataFrame(dfhistory)
df.to_csv(os.path.join(save_model_path, "history.csv"), header=True, index=False)

# Save best models to disk
torch.save(best_auc_model_state, os.path.join(save_model_path, f"max_AUC_HeteroTCR_epoch{best_auc_epoch:03d}_AUC{best_val_auc:.6f}.pth"))
torch.save(best_loss_model_state, os.path.join(save_model_path, f"minloss_AUC_HeteroTCR_epoch{best_loss_epoch:03d}_LOSS{best_val_loss:.6f}.pth"))

print("Best AUC model saved from epoch:", best_auc_epoch, "with AUC:", best_val_auc)
print("Best loss model saved from epoch:", best_loss_epoch, "with loss:", best_val_loss)


Train epoch: 1 - loss: 0.6934 - binary_accuracy: 0.4992 - ROCAUC: 0.4985 - val_loss: 0.6932 - val_binary_accuracy: 0.4997 - val_ROCAUC: 0.4995
Train epoch: 2 - loss: 0.6932 - binary_accuracy: 0.4992 - ROCAUC: 0.5032 - val_loss: 0.6931 - val_binary_accuracy: 0.5003 - val_ROCAUC: 0.5138
Train epoch: 3 - loss: 0.6931 - binary_accuracy: 0.5008 - ROCAUC: 0.5181 - val_loss: 0.6932 - val_binary_accuracy: 0.5003 - val_ROCAUC: 0.5207
Train epoch: 4 - loss: 0.6931 - binary_accuracy: 0.5008 - ROCAUC: 0.5247 - val_loss: 0.6931 - val_binary_accuracy: 0.5003 - val_ROCAUC: 0.5253
Train epoch: 5 - loss: 0.6931 - binary_accuracy: 0.5008 - ROCAUC: 0.5281 - val_loss: 0.6931 - val_binary_accuracy: 0.5003 - val_ROCAUC: 0.5268
Train epoch: 6 - loss: 0.6931 - binary_accuracy: 0.5008 - ROCAUC: 0.5298 - val_loss: 0.6931 - val_binary_accuracy: 0.5003 - val_ROCAUC: 0.5291
Train epoch: 7 - loss: 0.6931 - binary_accuracy: 0.5008 - ROCAUC: 0.5336 - val_loss: 0.6931 - val_binary_accuracy: 0.5209 - val_ROCAUC: 0.5320

KeyboardInterrupt: 

In [ ]:
data_hetero_train

HeteroData(
  tcr={ x=[490, 320] },
  peptide={ x=[116, 320] },
  (tcr, binds, epitope)={ edge_index=[2, 500] },
  (epitope, rev_binds, tcr)={ edge_index=[2, 500] },
  (tcr, similar, tcr)={ edge_index=[2, 1362] }
)

In [ ]:
model = GNN_model.HeteroTCR(data_hetero_test.metadata(), 256, 3, "SAGE")
model = model.to(device)
data_hetero_test = data_hetero_test.to(device)

with torch.no_grad():  # Initialize lazy modules.
    out = model(data_hetero_test.x_dict, data_hetero_test.edge_index_dict, test_edge_label_index)

model_dir = save_model_path
val_model_path = save_model_path
for root, dirs, files in os.walk(val_model_path):
    for file in files:
        if file.startswith("min"):
            PATH = os.path.join(val_model_path, file)
            model.load_state_dict(torch.load(PATH))
            test_loss, test_binary_accuracy, test_ROCAUC, test_prob = GNN_model.predict(model, data_hetero_test, test_edge_label_index, y_test)
            print("ACC: {:.4f}".format(test_binary_accuracy))
            print("AUC: {:.4f}".format(test_ROCAUC))

# Create a new dataframe with tcr, epitope, true label, and predicted binding probability
results_df = pd.DataFrame({
    'tcr': data_test['cdr3'],
    'epitope': data_test['peptide'],
    'true_label': y_test,
    'predicted_binding_prob': test_prob.cpu().numpy()
})

results_df.to_csv(f"{save_model_path}/results/min/test.csv", index=False)
f"{save_model_path}/results/min/test.csv"

ACC: 0.6555
AUC: 0.7372


/tmp/ipykernel_3400942/1008895924.py:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(PATH))


'/home/dsi/chenbis/repos/sol_lab/tcr_epitope_prediction_gnn_mst/output/models_new/0.08_mst_enrich/results/min/test.csv'

In [ ]:
model = GNN_model.HeteroTCR(data_hetero_test.metadata(), 256, 3, "SAGE")
model = model.to(device)
data_hetero_test = data_hetero_test.to(device)

with torch.no_grad():  # Initialize lazy modules.
    out = model(data_hetero_test.x_dict, data_hetero_test.edge_index_dict, test_edge_label_index)

model_dir = save_model_path
val_model_path = save_model_path
for root, dirs, files in os.walk(val_model_path):
    for file in files:
        if file.startswith("max"):
            PATH = os.path.join(val_model_path, file)
            model.load_state_dict(torch.load(PATH))
            test_loss, test_binary_accuracy, test_ROCAUC, test_prob = GNN_model.predict(model, data_hetero_test, test_edge_label_index, y_test)
            print("ACC: {:.4f}".format(test_binary_accuracy))
            print("AUC: {:.4f}".format(test_ROCAUC))

# Create a new dataframe with tcr, epitope, true label, and predicted binding probability
results_df = pd.DataFrame({
    'tcr': data_test['cdr3'],
    'epitope': data_test['peptide'],
    'true_label': y_test,
    'predicted_binding_prob': test_prob.cpu().numpy()
})

results_df.to_csv(f"{save_model_path}/results/max/test.csv", index=False)
f"{save_model_path}/results/max/test.csv"

ACC: 0.6555
AUC: 0.7372


/tmp/ipykernel_3400942/2526967415.py:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(PATH))


'/home/dsi/chenbis/repos/sol_lab/tcr_epitope_prediction_gnn_mst/output/models_new/0.08_mst_enrich/results/max/test.csv'